In [0]:
from pyspark.sql.functions import col, explode, when, trim, lower, regexp_extract
# --- Cell 1: Connect to the Bronze Layer ---
print("Connecting to the Lakehouse Bronze Layer...")

catalog_name = "bronze_dev"
schema_name = "api_sports"
bronze_table_name = "raw_serie_a_players_2024"

full_bronze_path = f"{catalog_name}.{schema_name}.{bronze_table_name}"

# Read the raw Delta table into a PySpark DataFrame
df_bronze = spark.read.table(full_bronze_path)

TARGET_LEAGUE_ID = (
    df_bronze
    .withColumn("stat", explode(col("statistics")))
    .select(col("stat.league.id").alias("league_id"))
    .first()["league_id"]
)
print(f"League ID dynamically resolved from Bronze table: {TARGET_LEAGUE_ID}")

TARGET_NATIONALITY = "colombia"  # Change this to filter any nationality in Gold (lowercase)

# Print the schema to inspect the nested JSON structures
print("\n--- Raw Data Schema ---")
df_bronze.printSchema()

# Display the actual data to visually inspect the columns
display(df_bronze)

In [0]:
print("Executing Generalized Silver Layer Transformations...")

# 1. EXPLODE
df_exploded = df_bronze.withColumn("stat_element", explode(col("statistics")))
df_exploded = df_exploded.filter(col("stat_element").isNotNull())

# 2. FLATTEN, TRIMMING STRING VALUES & RENAME
df_flattened = df_exploded.select(
    col("player.id").alias("player_id"),
    trim(col("player.name")).alias("player_name"),
    trim(col("player.firstname")).alias("first_name"),
    trim(col("player.lastname")).alias("last_name"),
    col("player.age").cast("integer").alias("age"),
    regexp_extract(col("player.height"), r"(\d+)", 1).cast("integer").alias("height_cm"),
    regexp_extract(col("player.weight"), r"(\d+)", 1).cast("integer").alias("weight_kg"),
    trim(lower(col("player.nationality"))).alias("nationality"),
    col("stat_element.team.id").alias("team_id"),
    trim(lower(col("stat_element.team.name"))).alias("team_name"),
    col("stat_element.league.id").alias("league_id"),
    trim(lower(col("stat_element.league.name"))).alias("league_name"),
    col("stat_element.league.season").cast("integer").alias("season"),
    col("stat_element.games.appearences").cast("integer").alias("appearances"),
    col("stat_element.games.lineups").cast("integer").alias("lineups"),
    col("stat_element.games.minutes").cast("integer").alias("minutes_played"),
    trim(lower(col("stat_element.games.position"))).alias("position"),
    col("stat_element.games.rating").cast("float").alias("rating"),
    col("stat_element.goals.total").cast("integer").alias("goals_total"),
    col("stat_element.goals.assists").cast("integer").alias("assists_total"),
    col("stat_element.shots.total").cast("integer").alias("shots_total"),
    col("stat_element.shots.on").cast("integer").alias("shots_on_target"),
    col("stat_element.passes.total").cast("integer").alias("passes_total"),
    col("stat_element.passes.key").cast("integer").alias("key_passes"),
    col("stat_element.passes.accuracy").cast("integer").alias("pass_accuracy_percent"),
    col("stat_element.dribbles.attempts").cast("integer").alias("dribble_attempts"),
    col("stat_element.dribbles.success").cast("integer").alias("dribbles_successful"),
    col("stat_element.duels.total").cast("integer").alias("duels_total"),
    col("stat_element.duels.won").cast("integer").alias("duels_won"),
    col("stat_element.tackles.total").cast("integer").alias("tackles_total"),
    col("stat_element.tackles.interceptions").cast("integer").alias("interceptions"),
    col("stat_element.tackles.blocks").cast("integer").alias("blocks"),
    col("stat_element.cards.yellow").cast("integer").alias("yellow_cards"),
    col("stat_element.cards.red").cast("integer").alias("red_cards"),
    col("stat_element.substitutes.in").cast("integer").alias("sub_in"),
    col("stat_element.substitutes.out").cast("integer").alias("sub_out"),
    col("stat_element.fouls.committed").cast("integer").alias("fouls_committed"),
    col("stat_element.fouls.drawn").cast("integer").alias("fouls_drawn"),
    col("stat_element.goals.conceded").cast("integer").alias("goals_conceded"),
    col("stat_element.goals.saves").cast("integer").alias("saves"),
    col("stat_element.penalty.saved").cast("integer").alias("penalties_saved"),
    col("stat_element.penalty.scored").cast("integer").alias("penalties_scored"),
    col("stat_element.penalty.missed").cast("integer").alias("penalties_missed")
)

# 3. FILTER: Dynamic League ID and removing "Ghost" players
df_league_only = df_flattened.filter(
    (col("league_id") == TARGET_LEAGUE_ID) & 
    (col("minutes_played") > 0) # Data Quality: Drop players who never played a single minute!
)

# 4. DEDUPLICATE
df_silver_clean = df_league_only.dropDuplicates(["player_id", "team_id", "season"])

# 5. HANDLE NULLS: Only fill CUMULATIVE stats with 0. 
# We intentionally leave 'pass_accuracy_percent' and 'rating' out of this list so they remain NULL.
cumulative_stats = [
    # Appearances & Time
    "appearances",
    "minutes_played",
    "lineups",
    "sub_in",
    "sub_out",
    
    # Attack
    "goals_total",
    "assists_total",
    "shots_total",
    "shots_on_target",
    "penalties_scored",
    "penalties_missed",
    
    # Passing
    "passes_total",
    "key_passes",
    
    # Dribbling
    "dribble_attempts",
    "dribbles_successful",
    
    # Duels & Defending
    "duels_total",
    "duels_won",
    "tackles_total",
    "interceptions",
    "blocks",
    
    # Fouls & Discipline
    "fouls_committed",
    "fouls_drawn",
    "yellow_cards",
    "red_cards",
    
    # Goalkeeper
    "goals_conceded",
    "saves",
    "penalties_saved"
]

df_silver_final = df_silver_clean.fillna(0, subset=cumulative_stats)

# Step 6 — after fillna, add derived columns
df_silver_final = df_silver_clean.fillna(0, subset=cumulative_stats).select(
    "*",  # keep all existing columns
    when(col("shots_total") > 0,
        (col("shots_on_target") / col("shots_total") * 100)
        .cast("float")).alias("shot_accuracy_percent"),
    when(col("dribble_attempts") > 0,
        (col("dribbles_successful") / col("dribble_attempts") * 100)
        .cast("float")).alias("dribble_success_rate"),
    when(col("duels_total") > 0,
        (col("duels_won") / col("duels_total") * 100)
        .cast("float")).alias("duel_win_rate"),
    when(col("shots_total") > 0,
        (col("goals_total") / col("shots_total") * 100)
        .cast("float")).alias("goal_conversion_rate"),
    when(col("goals_total") > 0,
        (col("minutes_played") / col("goals_total"))
        .cast("float")).alias("minutes_per_goal"),
)

# Add at the end of Cell 2, before Cell 3:
print("\n--- Transformation Summary ---")
print(f"Records after explode: {df_exploded.count()}")
print(f"Records after league filter + active players: {df_league_only.count()}")
print(f"Records after deduplication: {df_silver_clean.count()}")
print(f"Final Silver record count: {df_silver_final.count()}")
print(f"Nationalities represented: {df_silver_final.select('nationality').distinct().count()}")
print(f"Teams represented: {df_silver_final.select('team_name').distinct().count()}")

In [0]:
print("Provisioning Silver Layer Infrastructure & Saving Data...")

# 1. Define Silver Layer path parameters (From your Roadmap!)
silver_catalog = "silver_dev"
silver_schema = "football_analytics"
silver_table = "players_cleaned"

full_silver_path = f"{silver_catalog}.{silver_schema}.{silver_table}"

# 2. Ensure the Unity Catalog infrastructure exists for the Silver Layer
spark.sql(f"CREATE CATALOG IF NOT EXISTS {silver_catalog}")
spark.sql(f"USE CATALOG {silver_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# 3. Write the cleaned DataFrame to the Silver Delta table
print(f"Committing cleaned DataFrame to Delta Lake...")

(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_silver_path)
)

print(f"SUCCESS! Silver pipeline complete. Data is safely stored in: {full_silver_path}")

In [0]:
%sql
SELECT * FROM silver_dev.football_analytics.players_cleaned LIMIT 10